# Adding the indeterminates to the scale and deciding on the cutoff

---

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
from IPython.display import display
from sklearn.metrics import brier_score_loss, roc_auc_score

relative_dir = Path("data") / "KGB Modelling"

project_root = next(
    (
        folder
        for folder in [Path.cwd(), *Path.cwd().parents]
        if (folder / relative_dir / "KGB in sample.csv").is_file()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError("Could not locate data/KGB Modelling/KGB in sample.csv.")

data_dir = project_root / relative_dir

known_train = pd.read_csv(data_dir / "KGB in sample.csv")
known_test = pd.read_csv(data_dir / "KGB out sample.csv")
indeterminate = pd.read_csv(data_dir / "indeterminates.csv")

print(f"Project: {project_root}")
print(f"Known good/bad development: {len(known_train):,}")
print(f"Known good/bad out-of-sample: {len(known_test):,}")
print(f"Indeterminate: {len(indeterminate):,}")

Project: c:\Users\georg\OneDrive\Coding\Origination-Scorecard
Known good/bad development: 7,258
Known good/bad out-of-sample: 2,458
Indeterminate: 197


---

## Bridging over the datasets and merging them into a KIGB sample

In [2]:
# Notebook 06 used this date to split development and out-of-sample.
split_date = pd.Timestamp("2024-07-01")

for name, data in [
    ("known_train", known_train),
    ("known_test", known_test),
    ("indeterminate", indeterminate),
]:
    data["application_date"] = pd.to_datetime(data["application_date"], errors="raise")
    if data["application_date"].isna().any():
        raise ValueError(f"{name} contains a missing application date.")

if not (known_train["application_date"] < split_date).all():
    raise ValueError("Known in-sample file contains later applications.")
if not (known_test["application_date"] >= split_date).all():
    raise ValueError("Known out-of-sample file contains earlier applications.")

indeterminate_train = indeterminate.loc[indeterminate["application_date"] < split_date].copy()
indeterminate_test = indeterminate.loc[indeterminate["application_date"] >= split_date].copy()


def add_group(data, group):
    result = data.copy()
    result["source_group"] = group
    return result


kigb_train = pd.concat(
    [
        add_group(known_train, "Known good/bad"),
        add_group(indeterminate_train, "Indeterminate"),
    ],
    ignore_index=True,
)

kigb_test = pd.concat(
    [
        add_group(known_test, "Known good/bad"),
        add_group(indeterminate_test, "Indeterminate"),
    ],
    ignore_index=True,
)

all_ids = pd.concat(
    [kigb_train["account_id"], kigb_test["account_id"]],
    ignore_index=True,
)
if all_ids.isna().any() or all_ids.duplicated().any():
    raise ValueError("Missing or duplicate account IDs.")

# Define the outcome from observed 12-month payment history.
# This intentionally does not use a KGB score to assign labels.
for sample in [kigb_train, kigb_test]:
    missed = pd.to_numeric(sample["missed_payment_months"], errors="raise")
    if missed.isna().any() or missed.lt(0).any():
        raise ValueError("Invalid missed-payment count.")

    sample["kigb_bad"] = missed.ge(3).astype(int)
    sample["outcome_group"] = np.select(
        [missed.eq(0), missed.between(1, 2), missed.ge(3)],
        ["Known good", "Indeterminate", "Known bad"],
        default="Invalid",
    )

if not kigb_train["kigb_bad"].isin([0, 1]).all():
    raise ValueError("Invalid development target.")

display(
    pd.DataFrame(
        {
            "Development": kigb_train["outcome_group"].value_counts(),
            "Out-of-sample": kigb_test["outcome_group"].value_counts(),
        }
    )
    .fillna(0)
    .astype(int)
)

print(f"Development accounts: {len(kigb_train):,}")
print(f"Out-of-sample accounts: {len(kigb_test):,}")

,Development,Out-of-sample
outcome_group,,
Known good,6964,2356
Known bad,294,102
Indeterminate,147,50


Development accounts: 7,405
Out-of-sample accounts: 2,508


---

## Prepare the same regression as used on the KGB sample

In [3]:
scaled_predictors = [
    "debt_to_income_ratio",
    "recent_application_count",
    "revolving_utilisation",
    "historic_defaults",
    "log_address_tenure",
]

raw_numeric = [
    "debt_to_income_ratio",
    "recent_application_count",
    "revolving_utilisation",
    "historic_defaults",
    "disposable_income",
    "time_at_address_months",
]


def prepare_predictors(data):
    rows = data.copy()

    required = [*raw_numeric, "loan_purpose"]
    missing_columns = sorted(set(required) - set(rows.columns))
    if missing_columns:
        raise ValueError(f"Missing predictors: {missing_columns}")

    for column in raw_numeric:
        rows[column] = pd.to_numeric(rows[column], errors="raise")

    if rows[required].isna().any().any():
        raise ValueError("Missing predictor value.")
    if not np.isfinite(rows[raw_numeric].to_numpy()).all():
        raise ValueError("Infinite numeric predictor value.")
    if rows["time_at_address_months"].lt(0).any():
        raise ValueError("Negative address tenure.")

    rows["low_disposable_income_flag"] = (rows["disposable_income"] < 150).astype(float)
    rows["log_address_tenure"] = np.log1p(rows["time_at_address_months"])

    return rows


kigb_train = prepare_predictors(kigb_train)
kigb_test = prepare_predictors(kigb_test)

# Learn scaling from the combined development sample only.
means = kigb_train[scaled_predictors].mean()
stds = kigb_train[scaled_predictors].std(ddof=0)

if not np.isfinite(stds).all() or (stds == 0).any():
    raise ValueError("A predictor has invalid or zero variation.")


def design_matrix(rows):
    x = (rows[scaled_predictors] - means) / stds
    x.insert(0, "const", 1.0)
    x["low_disposable_income_flag"] = rows["low_disposable_income_flag"]
    x["loan_purpose=education"] = (rows["loan_purpose"] == "education").astype(float)
    return x.astype(float)


x_train = design_matrix(kigb_train)
x_test = design_matrix(kigb_test)
y_train = kigb_train["kigb_bad"]
y_test = kigb_test["kigb_bad"]

print("Development design matrix:", x_train.shape)
print("Out-of-sample design matrix:", x_test.shape)

Development design matrix: (7405, 8)
Out-of-sample design matrix: (2508, 8)


---

## Fit KIGB Regression

In [4]:
if np.linalg.matrix_rank(x_train.to_numpy()) < x_train.shape[1]:
    raise ValueError("Development predictors are perfectly collinear.")

kigb_model = sm.Logit(y_train, x_train).fit(maxiter=200, disp=False)

if not kigb_model.mle_retvals.get("converged", False):
    raise RuntimeError("KIGB logistic regression did not converge.")

print(kigb_model.summary())

kigb_train_pd = pd.Series(
    kigb_model.predict(x_train).to_numpy(),
    index=kigb_train.index,
    name="predicted_pd",
)
kigb_test_pd = pd.Series(
    kigb_model.predict(x_test).to_numpy(),
    index=kigb_test.index,
    name="predicted_pd",
)

                           Logit Regression Results                           
Dep. Variable:               kigb_bad   No. Observations:                 7405
Model:                          Logit   Df Residuals:                     7397
Method:                           MLE   Df Model:                            7
Date:                Fri, 02 Oct 2026   Pseudo R-squ.:                 0.02636
Time:                        11:54:05   Log-Likelihood:                -1204.0
converged:                       True   LL-Null:                       -1236.6
Covariance Type:            nonrobust   LLR p-value:                 1.379e-11
                                 coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------
const                         -3.3318      0.075    -44.473      0.000      -3.479      -3.185
debt_to_income_ratio           0.1047      0.058      1.811      0.070      -0.009 

---

## Apply the existing PD Log odds scoring scale

In [6]:
def model_results(actual, predicted_pd):
    return {
        "Accounts": len(actual),
        "Observed bad rate": actual.mean(),
        "Mean predicted PD": predicted_pd.mean(),
        "ROC AUC": roc_auc_score(actual, predicted_pd),
        "Brier score": brier_score_loss(actual, predicted_pd),
    }


results = pd.DataFrame(
    {
        "Development": model_results(y_train, kigb_train_pd),
        "Out-of-sample": model_results(y_test, kigb_test_pd),
    }
)
display(results.round(4))

# Keep the same scale: 600 points at 50:1 good:bad odds,
# with 20 points for each doubling of good:bad odds.
base_score = 600
base_good_bad_odds = 50
points_to_double_odds = 20

factor = points_to_double_odds / np.log(2)
offset = base_score - factor * np.log(base_good_bad_odds)


def pd_to_score(probability):
    p = np.asarray(probability, dtype=float)
    if not np.isfinite(p).all() or ((p < 0) | (p > 1)).any():
        raise ValueError("Invalid predicted PD.")
    p = np.clip(p, 1e-12, 1 - 1e-12)
    return offset + factor * np.log((1 - p) / p)


kigb_train["predicted_pd"] = kigb_train_pd
kigb_train["score"] = pd_to_score(kigb_train_pd)

kigb_test["predicted_pd"] = kigb_test_pd
kigb_test["score"] = pd_to_score(kigb_test_pd)

display(
    kigb_test[["account_id", "outcome_group", "predicted_pd", "score"]]
    .head(1)
    .style.format(
        {
            "predicted_pd": "{:.2%}",
            "score": "{:.1f}",
        }
    )
)

,Development,Out-of-sample
Accounts,7405.0000,2508.0000
Observed bad rate,0.0397,0.0407
Mean predicted PD,0.0397,0.0383
ROC AUC,0.6385,0.6429
Brier score,0.0378,0.0386


,account_id,outcome_group,predicted_pd,score
0,ACC-10000008,Known good,3.42%,583.5


---

## Saving the datasets

In [7]:
in_sample_path = data_dir / "KIGB in sample.csv"
out_sample_path = data_dir / "KIGB out sample.csv"

required_output = {
    "account_id",
    "application_date",
    "outcome_group",
    "kigb_bad",
    "predicted_pd",
    "score",
}

for name, data in [
    ("KIGB in-sample", kigb_train),
    ("KIGB out-of-sample", kigb_test),
]:
    missing = required_output - set(data.columns)
    if missing:
        raise ValueError(f"{name} is missing: {sorted(missing)}")

kigb_train.to_csv(in_sample_path, index=False)
kigb_test.to_csv(out_sample_path, index=False)

print(f"Saved {len(kigb_train):,} accounts: {in_sample_path}")
print(f"Saved {len(kigb_test):,} accounts: {out_sample_path}")

Saved 7,405 accounts: c:\Users\georg\OneDrive\Coding\Origination-Scorecard\data\KGB Modelling\KIGB in sample.csv
Saved 2,508 accounts: c:\Users\georg\OneDrive\Coding\Origination-Scorecard\data\KGB Modelling\KIGB out sample.csv


---

## Veiwing the new PD x Score scatter

---

# Establishing the cutoff

In [8]:
relative_dir = Path("data") / "KGB Modelling"
project_root = next(
    (
        folder
        for folder in [Path.cwd(), *Path.cwd().parents]
        if (folder / relative_dir / "KIGB in sample.csv").is_file()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError("Could not locate data/KGB Modelling/KIGB in sample.csv.")

data_dir = project_root / relative_dir
samples = {
    "In-sample": pd.read_csv(data_dir / "KIGB in sample.csv"),
    "Out-of-sample": pd.read_csv(data_dir / "KIGB out sample.csv"),
}

required = {
    "account_id",
    "outcome_group",
    "kigb_bad",
    "predicted_pd",
    "score",
}

for sample_name, data in samples.items():
    missing = required - set(data.columns)
    if missing:
        raise ValueError(f"{sample_name} is missing {sorted(missing)}")

    for column in ["kigb_bad", "predicted_pd", "score"]:
        data[column] = pd.to_numeric(data[column], errors="raise")

    if data[list(required - {"account_id", "outcome_group"})].isna().any().any():
        raise ValueError(f"{sample_name} has missing model results.")
    if not data["kigb_bad"].isin([0, 1]).all():
        raise ValueError(f"{sample_name} has invalid bad labels.")
    if not data["predicted_pd"].between(0, 1).all():
        raise ValueError(f"{sample_name} has invalid PDs.")

    print(f"{sample_name}: {len(data):,} accounts")

In-sample: 7,405 accounts
Out-of-sample: 2,508 accounts


---

## Defining the cutoff range

In [9]:
# These are KIGB predicted-PD cutoffs.
# 3.69% is included as the numerical benchmark from the KGB model.
pd_cutoffs = sorted(
    set(
        [
            0.0100,
            0.0150,
            0.0200,
            0.0250,
            0.0300,
            0.0369,
            0.0400,
            0.0500,
            0.0600,
            0.0800,
            0.1000,
        ]
    )
)

target_book_bad_rate = 0.02
kgb_benchmark = 0.0369

# Convert each PD cutoff to a score on your existing scale.
base_score = 600
base_good_bad_odds = 50
points_to_double_odds = 20

factor = points_to_double_odds / np.log(2)
offset = base_score - factor * np.log(base_good_bad_odds)


def pd_to_score(probability):
    p = np.asarray(probability, dtype=float)
    if not np.isfinite(p).all() or ((p <= 0) | (p >= 1)).any():
        raise ValueError("Cutoff PD must be strictly between 0 and 1.")
    return offset + factor * np.log((1 - p) / p)


def calibration_measures(data):
    """Compare predicted PD with the observed 3+ missed-month rate."""
    if data.empty:
        return {
            "Accounts": 0,
            "Expected bads": np.nan,
            "Observed bads": np.nan,
            "Predicted bad rate": np.nan,
            "Observed bad rate": np.nan,
            "Calibration gap": np.nan,
            "Observed / expected": np.nan,
            "Brier score": np.nan,
        }

    expected_bads = data["predicted_pd"].sum()
    observed_bads = data["kigb_bad"].sum()
    predicted_rate = data["predicted_pd"].mean()
    observed_rate = data["kigb_bad"].mean()

    return {
        "Accounts": len(data),
        "Expected bads": expected_bads,
        "Observed bads": int(observed_bads),
        "Predicted bad rate": predicted_rate,
        "Observed bad rate": observed_rate,
        "Calibration gap": observed_rate - predicted_rate,
        "Observed / expected": (observed_bads / expected_bads if expected_bads > 0 else np.nan),
        "Brier score": brier_score_loss(data["kigb_bad"], data["predicted_pd"]),
    }

In [10]:
rows = []

for sample_name, data in samples.items():
    is_indeterminate = data["outcome_group"].eq("Indeterminate")

    for cutoff in pd_cutoffs:
        # Below the PD cutoff = accepted / inferred good.
        # At or above it = flagged / inferred bad.
        accepted = data.loc[data["predicted_pd"] < cutoff]
        flagged = data.loc[data["predicted_pd"] >= cutoff]

        accepted_metrics = calibration_measures(accepted)
        flagged_metrics = calibration_measures(flagged)

        rows.append(
            {
                "Sample": sample_name,
                "PD cutoff": cutoff,
                "Minimum score": float(pd_to_score(cutoff)),
                "Benchmark": ("Old KGB 3.69%" if np.isclose(cutoff, kgb_benchmark) else ""),
                "Approval rate": len(accepted) / len(data),
                "Accepted accounts": accepted_metrics["Accounts"],
                "Expected accepted bads": accepted_metrics["Expected bads"],
                "Observed accepted bads": accepted_metrics["Observed bads"],
                "Predicted accepted bad rate": accepted_metrics["Predicted bad rate"],
                "Observed accepted bad rate": accepted_metrics["Observed bad rate"],
                "Accepted calibration gap": accepted_metrics["Calibration gap"],
                "Accepted observed / expected": accepted_metrics["Observed / expected"],
                "Accepted Brier score": accepted_metrics["Brier score"],
                "Flagged accounts": flagged_metrics["Accounts"],
                "Predicted flagged bad rate": flagged_metrics["Predicted bad rate"],
                "Observed flagged bad rate": flagged_metrics["Observed bad rate"],
                "Inferred good": int((is_indeterminate & data["predicted_pd"].lt(cutoff)).sum()),
                "Inferred bad": int((is_indeterminate & data["predicted_pd"].ge(cutoff)).sum()),
            }
        )

cutoff_results = pd.DataFrame(rows)

display(
    cutoff_results.style.format(
        {
            "PD cutoff": "{:.2%}",
            "Minimum score": "{:.1f}",
            "Approval rate": "{:.1%}",
            "Expected accepted bads": "{:.1f}",
            "Predicted accepted bad rate": "{:.2%}",
            "Observed accepted bad rate": "{:.2%}",
            "Accepted calibration gap": "{:+.2%}",
            "Accepted observed / expected": "{:.2f}",
            "Accepted Brier score": "{:.4f}",
            "Predicted flagged bad rate": "{:.2%}",
            "Observed flagged bad rate": "{:.2%}",
        },
        na_rep="—",
    )
)

,Sample,PD cutoff,Minimum score,Benchmark,Approval rate,Accepted accounts,Expected accepted bads,Observed accepted bads,Predicted accepted bad rate,Observed accepted bad rate,Accepted calibration gap,Accepted observed / expected,Accepted Brier score,Flagged accounts,Predicted flagged bad rate,Observed flagged bad rate,Inferred good,Inferred bad
0,In-sample,1.00%,619.7,,1.1%,78,0.7,0,0.88%,0.00%,-0.88%,0.00,0.0001,7327,4.00%,4.01%,1,146
1,In-sample,1.50%,607.9,,2.9%,212,2.3,2,1.09%,0.94%,-0.15%,0.86,0.0093,7193,4.06%,4.06%,2,145
2,In-sample,2.00%,599.4,,5.9%,438,6.4,6,1.47%,1.37%,-0.10%,0.93,0.0135,6967,4.13%,4.13%,3,144
3,In-sample,2.50%,592.8,,17.2%,1272,25.5,28,2.00%,2.20%,+0.20%,1.10,0.0215,6133,4.38%,4.34%,12,135
4,In-sample,3.00%,587.4,,34.3%,2543,60.4,55,2.37%,2.16%,-0.21%,0.91,0.0212,4862,4.80%,4.92%,33,114
5,In-sample,3.69%,581.2,Old KGB 3.69%,56.2%,4164,114.3,105,2.75%,2.52%,-0.22%,0.92,0.0246,3241,5.54%,5.83%,60,87
6,In-sample,4.00%,578.8,,63.9%,4731,136.1,134,2.88%,2.83%,-0.04%,0.98,0.0274,2674,5.91%,5.98%,67,80
7,In-sample,5.00%,572.1,,79.2%,5867,186.6,183,3.18%,3.12%,-0.06%,0.98,0.0301,1538,6.98%,7.22%,101,46
8,In-sample,6.00%,566.5,,87.9%,6510,221.8,223,3.41%,3.43%,+0.02%,1.01,0.0329,895,8.07%,7.93%,117,30
9,In-sample,8.00%,557.6,,95.6%,7078,260.7,263,3.68%,3.72%,+0.03%,1.01,0.0355,327,10.17%,9.48%,136,11


---

## Finding cutoffs that predict a book bad rate of 2% or less

In [11]:
within_target = cutoff_results.loc[
    cutoff_results["Predicted accepted bad rate"] <= target_book_bad_rate
].copy()

print("Cutoffs with a predicted accepted bad rate of 2% or less:")
display(
    within_target[
        [
            "Sample",
            "PD cutoff",
            "Minimum score",
            "Approval rate",
            "Predicted accepted bad rate",
            "Observed accepted bad rate",
            "Accepted calibration gap",
            "Inferred good",
            "Inferred bad",
        ]
    ].style.format(
        {
            "PD cutoff": "{:.2%}",
            "Minimum score": "{:.1f}",
            "Approval rate": "{:.1%}",
            "Predicted accepted bad rate": "{:.2%}",
            "Observed accepted bad rate": "{:.2%}",
            "Accepted calibration gap": "{:+.2%}",
        }
    )
)

print("\nOverall calibration, before applying any cutoff:")
overall_calibration = pd.DataFrame(
    {name: calibration_measures(data) for name, data in samples.items()}
)
display(overall_calibration.round(4))

Cutoffs with a predicted accepted bad rate of 2% or less:


,Sample,PD cutoff,Minimum score,Approval rate,Predicted accepted bad rate,Observed accepted bad rate,Accepted calibration gap,Inferred good,Inferred bad
0,In-sample,1.00%,619.7,1.1%,0.88%,0.00%,-0.88%,1,146
1,In-sample,1.50%,607.9,2.9%,1.09%,0.94%,-0.15%,2,145
2,In-sample,2.00%,599.4,5.9%,1.47%,1.37%,-0.10%,3,144
11,Out-of-sample,1.00%,619.7,1.4%,0.87%,0.00%,-0.87%,0,50
12,Out-of-sample,1.50%,607.9,3.5%,1.09%,1.15%,+0.06%,1,49
13,Out-of-sample,2.00%,599.4,6.3%,1.42%,1.90%,+0.48%,2,48
14,Out-of-sample,2.50%,592.8,18.3%,1.98%,1.53%,-0.46%,8,42



Overall calibration, before applying any cutoff:


,In-sample,Out-of-sample
Accounts,7405.0000,2508.0000
Expected bads,294.0000,96.0336
Observed bads,294.0000,102.0000
Predicted bad rate,0.0397,0.0383
Observed bad rate,0.0397,0.0407
Calibration gap,0.0000,0.0024
Observed / expected,1.0000,1.0621
Brier score,0.0378,0.0386


---

I have chosen to use the 2.5% PD cutoff - it maximises the number of accepted cases while still remaining inside our goal 2% predicted book bad rate, need to now convert this into a score cutoff